In [3]:
###%pip install ISLP scikit-learn

from ISLP import load_data
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Load Boston dataset
Boston = load_data('Boston')
X = Boston[['lstat']].values
y = Boston['medv'].values

# Split into training and validation sets (253 each)
X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=253, test_size=253, random_state=42)

mse_list = []
for degree in range(1, 5):
	poly = PolynomialFeatures(degree)
	X_train_poly = poly.fit_transform(X_train)
	X_val_poly = poly.transform(X_val)
	model = LinearRegression().fit(X_train_poly, y_train)
	y_pred = model.predict(X_val_poly)
	mse = mean_squared_error(y_val, y_pred)
	mse_list.append(round(mse, 2))

result = np.array(mse_list)
print(result)



[38.51 30.84 29.22 27.75]


In [5]:
# Install ISLP if not already available
# %pip install ISLP scikit-learn

from ISLP import load_data
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate

# Load College dataset
College = load_data('College')
X = College[['Room.Board']].values
y = College['Outstate'].values
n = len(X)

mse_list = []
for degree in range(1, 6):
	poly = PolynomialFeatures(degree)
	X_poly = poly.fit_transform(X)
	model = LinearRegression()
	cv_results = cross_validate(model, X_poly, y, cv=n, scoring='neg_mean_squared_error', return_train_score=False)
	mse = -np.mean(cv_results['test_score'])
	mse_list.append(round(mse, 2))

result = np.array(mse_list)
print(result)


[9291471.1  9255509.68 9263314.52 9268010.35 9284911.37]


In [7]:
from sklearn.model_selection import KFold, cross_val_score

degrees = [1, 2, 3]
cv_errors_5 = []
cv_errors_10 = []

for d in degrees:
	poly = PolynomialFeatures(degree=d)
	X_poly = poly.fit_transform(X)
	model = LinearRegression()
	
	# 5-fold CV
	kf5 = KFold(n_splits=5, shuffle=True, random_state=123)
	scores_5 = cross_val_score(model, X_poly, y, cv=kf5, scoring='neg_mean_squared_error')
	cv_errors_5.append(round(-scores_5.mean(), 3))
	
	# 10-fold CV
	kf10 = KFold(n_splits=10, shuffle=True, random_state=123)
	scores_10 = cross_val_score(model, X_poly, y, cv=kf10, scoring='neg_mean_squared_error')
	cv_errors_10.append(round(-scores_10.mean(), 3))

# Report results
for i, d in enumerate(degrees):
	print(f"Degree {d}: 5-fold CV error = {cv_errors_5[i]}, 10-fold CV error = {cv_errors_10[i]}, Difference = {round(cv_errors_5[i] - cv_errors_10[i], 3)}")



Degree 1: 5-fold CV error = 9377937.163, 10-fold CV error = 9315542.59, Difference = 62394.573
Degree 2: 5-fold CV error = 9345970.983, 10-fold CV error = 9281464.281, Difference = 64506.702
Degree 3: 5-fold CV error = 9385687.25, 10-fold CV error = 9293767.855, Difference = 91919.395


In [ ]:
# Load Default dataset
Default = load_data('Default')

# Function to compute correlation coefficient
def corr_balance_income(data):
	return np.corrcoef(data['balance'], data['income'])[0, 1]

# Bootstrap resampling
B = 2000
rng = np.random.RandomState(456)
n = len(Default)
boot_corrs = []
for _ in range(B):
	sample_idx = rng.choice(n, n, replace=True)
	sample = Default.iloc[sample_idx]
	boot_corrs.append(corr_balance_income(sample))

boot_se = np.std(boot_corrs)
actual_corr = corr_balance_income(Default)

print(f"Bootstrap SE: {boot_se:.4f}")
print(f"Actual correlation: {actual_corr:.4f}")
print(f"Bootstrap correlations: {boot_corrs[:2]}")  ### report first 2 - of course it can contiune to 2000


Bootstrap SE: 0.0099
Actual correlation: -0.1522
Bootstrap correlations: [np.float64(-0.16052424900639561), np.float64(-0.1586607374227576)]


In [9]:
# Load Wage dataset
from ISLP import load_data
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

# Simple linear regression (age only)
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm


Wage = load_data('Wage')
X = Wage[['age']].values
y = Wage['wage'].values


# Fit linear regression model
X_lin = Wage[['age']].values
model_lin = LinearRegression().fit(X_lin, y)
ols_coef_lin = model_lin.coef_[0]
ols_intercept_lin = model_lin.intercept_

# OLS standard errors using statsmodels
X_sm_lin = sm.add_constant(X_lin)
ols_model_lin = sm.OLS(y, X_sm_lin).fit()
ols_se_lin = ols_model_lin.bse  # [intercept, age]

# Bootstrap standard errors for linear regression
B = 1500
rng = np.random.RandomState(789)
n = len(y)
boot_coefs_lin = np.zeros((B, 2))
for i in range(B):
    idx = rng.choice(n, n, replace=True)
    Xb = X_lin[idx]
    yb = y[idx]
    m = LinearRegression().fit(Xb, yb)
    boot_coefs_lin[i] = [m.intercept_, m.coef_[0]]

boot_se_lin = boot_coefs_lin.std(axis=0)


# Fit quadratic model
poly2 = PolynomialFeatures(degree=2, include_bias=True)
X_poly2 = poly2.fit_transform(X)
model2 = LinearRegression().fit(X_poly2, y)
ols_coefs = model2.coef_
ols_intercept = model2.intercept_

# Get OLS standard errors using statsmodels for summary
import statsmodels.api as sm
X_sm = sm.add_constant(np.column_stack([X, X**2]))
ols_model = sm.OLS(y, X_sm).fit()
ols_se = ols_model.bse  # [intercept, age, age^2]

# Bootstrap standard errors
B = 1500
rng = np.random.RandomState(789)
n = len(y)
boot_coefs = np.zeros((B, 3))
for i in range(B):
	idx = rng.choice(n, n, replace=True)
	Xb = X[idx]
	yb = y[idx]
	Xb_poly2 = poly2.transform(Xb)
	m = LinearRegression().fit(Xb_poly2, yb)
	boot_coefs[i] = [m.intercept_, m.coef_[1], m.coef_[2]]

boot_se = boot_coefs.std(axis=0)


# Ratio of bootstrap SE to OLS SE, rounded to 3 decimals
ratios_lin = np.round(boot_se_lin / ols_se_lin, 3)
print("Linear model coefficients (intercept, age):")
print("Bootstrap SE:", np.round(boot_se_lin, 3))
print("OLS SE:", np.round(ols_se_lin, 3))
print("Ratio (Bootstrap SE / OLS SE):", ratios_lin)

# Ratio of bootstrap SE to OLS SE, rounded to 3 decimals
ratios = np.round(boot_se / ols_se, 3)
print("Quadratic model coefficients (intercept, age, age^2):")
print("Bootstrap SE:", np.round(boot_se, 3))
print("OLS SE:", np.round(ols_se, 3))
print("Ratio (Bootstrap SE / OLS SE):", ratios)



### for some reason the SE for the quadratic is rounding to only 1 decimal place. 

Linear model coefficients (intercept, age):
Bootstrap SE: [2.577 0.062]
OLS SE: [2.846 0.065]
Ratio (Bootstrap SE / OLS SE): [0.905 0.961]
Quadratic model coefficients (intercept, age, age^2):
Bootstrap SE: [6.166e+00 3.150e-01 4.000e-03]
OLS SE: [8.19e+00 3.89e-01 4.00e-03]
Ratio (Bootstrap SE / OLS SE): [0.753 0.811 0.848]
